# Lab 07-03 — LLM-as-Judge vs Reference Metrics: Cohen's Kappa

**Track 07 · Evaluation** — before you trust an LLM judge, prove it agrees with a reference-based rater.

LLM-as-judge is convenient and flexible, but it is a model — it can be lenient, blind to a topic, or unstable. This lab cross-checks it against a deterministic reference rater (answer↔gold cosine) on the same 15 questions and measures the agreement with **Cohen's kappa**, the agreement statistic that subtracts chance agreement.

```text
same pipeline as lab 02 (retrieve -> generate)
  -> LLM judge verdict  (correct / incorrect)
  -> reference verdict  (cosine >= threshold)
  -> observed agreement + Cohen's kappa
  -> verification gate (--verify)
```

The lesson generalizes to the repo's LLM-as-judge runs: they are only trusted after a kappa cross-check against reference metrics.


## Setup

This notebook mirrors `src/curriculum/07-evaluation/03-judge-vs-reference.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

From the terminal, the lab runs as:

```bash
python src/curriculum/07-evaluation/03-judge-vs-reference.py          # run + demo
python src/curriculum/07-evaluation/03-judge-vs-reference.py --verify # verification gate
```

**LLM keys**: the lab generates with GroqLLM and judges with the local Ollama judge — `GROQ_API_KEY` must be in the repo-root `.env` (the imports cell loads it).

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the generator, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))


import pandas as pd  # noqa: E402
from dotenv import load_dotenv  # noqa: E402
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env
from embeddings.bge import BGEEmbedding  # noqa: E402
from evaluation.harness import EvaluationHarness  # noqa: E402
from evaluation.judge import LLMJudge  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402



## 1. Configuration

Same corpus, same sample size, same `top_k` as lab 02 — this lab changes only *how the answer is scored*. `COSINE_THRESHOLD = 0.5` is the reference rater's decision boundary; the judge rater is the local Ollama `LLMJudge`.


In [ ]:
# 1. Configuration
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
TEST_PATH = RAG_MINI / "test.parquet"
N_SAMPLE = 15
TOP_K = 3
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
COSINE_THRESHOLD = 0.5  # reference rater: cosine >= this => "correct"


## 2. Load

The same rag-mini loader as lab 02: the passages parquet (single `passage` column) plus the test QA.


In [ ]:
# 2. Load
def load_passages(path: Path) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for every passage."""
    df = pd.read_parquet(path)
    texts = [str(row["passage"]).strip() for _, row in df.iterrows()]
    return texts, [str(i) for i in range(len(df))]


def load_test_qa(path: Path) -> list[dict]:
    """Return [{"question": ..., "answer": ...}] from test.parquet."""
    df = pd.read_parquet(path)
    return [{"question": r["question"], "answer": r["answer"]}
            for _, r in df.iterrows()]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 3. The two raters

Two independent verdict functions over the same (question, answer, reference) triple:

- **judge_verdict** — the LLM judge reads the question, answer and gold reference and returns `correct` / `incorrect` as JSON.
- **reference_verdict** — deterministic: cosine(answer, reference) ≥ 0.5 means correct.

Neither is "ground truth" — that is the point. They are two fallible measurement instruments, and we are about to measure how well they agree.


In [ ]:
# 3. The two raters
def judge_verdict(judge: LLMJudge, question: str, answer: str,
                  reference: str) -> int:
    """LLM judge: is the answer correct? -> 1 correct, 0 incorrect.

    JSON schema ``{"correct": true|false}``; accepts bool or "yes"/"true"
    strings from the model.
    """
    instruction = (
        "You are an answer-quality judge. Decide whether the answer is "
        "CORRECT given the gold reference. Output ONLY JSON."
    )
    prompt = (
        f"Question: {question}\n\nAnswer: {answer}\n\n"
        f"Gold reference: {reference}\n\n"
        'Return JSON: {"correct": true} or {"correct": false}'
    )
    result = judge.judge(instruction, prompt)
    raw = result.get("correct", False)
    if isinstance(raw, bool):
        return 1 if raw else 0
    return 1 if str(raw).strip().lower() in ("yes", "true", "1", "correct") else 0


def reference_verdict(cosine: float) -> int:
    """Reference-based rater: cosine against gold >= threshold -> 1 else 0."""
    return 1 if cosine >= COSINE_THRESHOLD else 0


## 4. Experiment

The pipeline is identical to lab 02's (embed on CPU — the shared GPU is held by the local Ollama judge — retrieve, generate), but each answer now gets *both* verdicts. `EvaluationHarness.kappa` computes Cohen's kappa from the two label lists.


In [ ]:
# 4. Experiment
def run_experiment() -> dict:
    passages, passage_ids = load_passages(PASSAGES_PATH)
    test_qa = load_test_qa(TEST_PATH)

    # device="cpu": the local Ollama judge holds the shared GPU (5.6 GiB
    # here); bulk-embedding 3200 passages on CPU avoids CUDA OOM and leaves
    # the GPU for the judge calls that follow.
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device="cpu")
    t0 = time.perf_counter()
    vectors = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(passages, passage_ids)
    ]
    store = FAISSVectorStore(embedding=embedder)
    store.add(chunks, embeddings=vectors)
    retriever = SimilarityRetriever(store, top_k=TOP_K)

    llm = GroqLLM(temperature=0.0)
    judge = LLMJudge()

    judge_labels: list[int] = []
    reference_labels: list[int] = []
    rows: list[dict] = []

    t0 = time.perf_counter()
    for item in test_qa[:N_SAMPLE]:
        question, reference = item["question"], item["answer"]
        context = "\n\n".join(d.page_content
                              for d in retriever.retrieve(question))
        answer = llm.invoke(
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            "Answer in one or two complete sentences, stating the key "
            "fact(s) from the context:"
        ).strip()

        cosine = cosine_similarity(judge.embed([answer])[0],
                                   judge.embed([reference])[0])
        jv = judge_verdict(judge, question, answer, reference)
        rv = reference_verdict(cosine)
        judge_labels.append(jv)
        reference_labels.append(rv)
        rows.append({
            "question": question,
            "reference": reference,
            "answer": answer,
            "cosine": cosine,
            "judge": jv,
            "reference": rv,
        })
    llm_s = time.perf_counter() - t0

    kappa = EvaluationHarness.kappa(judge_labels, reference_labels)
    n = len(judge_labels)
    agreement = sum(1 for a, b in zip(judge_labels, reference_labels)
                    if a == b) / n if n else 0.0
    return {
        "rows": rows,
        "indexed": len(passages),
        "embed_s": embed_s,
        "llm_s": llm_s,
        "kappa": kappa,
        "agreement": agreement,
        "judge_labels": judge_labels,
        "reference_labels": reference_labels,
    }


def landis_koch(kappa: float) -> str:
    """Human words for a kappa value (Landis & Koch, 1977)."""
    if kappa < 0.0:
        return "poor (worse than chance)"
    if kappa < 0.21:
        return "slight"
    if kappa < 0.41:
        return "fair"
    if kappa < 0.61:
        return "moderate"
    if kappa < 0.81:
        return "substantial"
    return "almost perfect"


## 5. Demo

The demo prints the per-question verdict pairs (AGREE / DISAGREE) and the headline numbers. Expect a familiar pattern: observed agreement looks decent (~0.6) while kappa is far lower (~0.1) — the judge is lenient and both raters say "correct" most of the time, so they agree by chance. Kappa is the honest number.


In [ ]:
# 5. Demo
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-03 — LLM-as-judge vs reference: Cohen's kappa")
    print(f"rag-mini {exp['indexed']} passages, {len(exp['rows'])} questions")
    print("=" * 66)

    print("\n[1] Verdicts per question (judge vs reference-based):")
    for row in exp["rows"]:
        mark = "AGREE" if row["judge"] == row["reference"] else "DISAGREE"
        print(f"    judge={row['judge']} ref={row['reference']} "
              f"cos={row['cosine']:.2f} [{mark}] {row['question'][:50]}")

    k = exp["kappa"]
    print(f"\n[2] Agreement: observed {exp['agreement']:.3f}, "
          f"Cohen's kappa = {k:.3f} -> {landis_koch(k)}")

    print(f"\n[3] Takeaway")
    print("    Observed agreement alone flatters: two raters who both say")
    print("    'correct' most of the time agree by chance. Kappa subtracts")
    print("    that expected agreement, so it is the honest number. A kappa")
    print("    below ~0.6 means the LLM judge is not interchangeable with")
    print("    the reference metric — trust it only after this cross-check,")
    print("    and prefer it as a supplement, not a replacement. Note the")
    print("    threshold choice (cosine >= 0.5) also moves the reference")
    print("    rater's side of the table: kappa audits the PAIR.")


## 6. Verification gate

The `--verify` gate checks both raters produced a verdict per question, kappa is finite and in range, and — the interesting checks — the reference rater found *some* correct answers while the judge is *not* a yes-machine (it produced at least one incorrect verdict).


In [ ]:
# 6. Verification gate
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    n = len(exp["rows"])
    k = exp["kappa"]

    checks.append((f"{N_SAMPLE} questions judged (>= 10)", n >= 10))
    checks.append(("kappa is finite and in [-1, 1]",
                   -1.0 <= k <= 1.0 and k == k))
    checks.append(("both verdict lists have one label per question",
                   len(exp["judge_labels"]) == n
                   and len(exp["reference_labels"]) == n))
    checks.append(("reference rater found some correct answers (sum > 0)",
                   sum(exp["reference_labels"]) > 0))
    checks.append(("LLM judge produced some incorrect verdicts (sum < n) "
                   "- judge is not a yes-machine",
                   sum(exp["judge_labels"]) < n))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Bulk-embedding on CPU takes a few minutes, then 15 generations + judge calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
